# SOC Analyst environment in Google Colab

This notebook installs the [Soc Analyst](https://huggingface.co/spaces/Suryaai05/Soc-Analyst-Final-Repo) FastAPI app, runs it in the background, and calls the API from Colab (localhost works **inside the same runtime**).

**Runtime:** use **GPU** if you plan PPO training (optional). First startup can take **a few minutes** (PyTorch, transformers, OpenEnv import).

**Repo:** set `REPO_URL` to your GitHub or Hugging Face Git URL (public clone).

In [ ]:
# --- Configure clone URL (pick one) ---
REPO_URL = "https://huggingface.co/spaces/Suryaai05/Soc-Analyst-Final-Repo"  # or "https://github.com/Ruthvik4257/Soc-Analyst-Final-Repo.git"
REPO_DIR = "/content/soc-analyst-env"

import os
import subprocess
from pathlib import Path

if not Path(REPO_DIR).is_dir():
  subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
  print("Repo already present; delete", REPO_DIR, "to re-clone fresh.")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# Install dependencies (Colab has torch often; this aligns the rest with requirements.txt)
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
# Expose app root on PYTHONPATH and start Uvicorn in the background
import os
import subprocess
import time
import sys
from pathlib import Path

root = str(Path(os.getcwd()).resolve())
os.environ["PYTHONPATH"] = root + os.pathsep + os.environ.get("PYTHONPATH", "")
PORT = "8000"

logf = open("/tmp/uvicorn_soc.log", "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", PORT],
    cwd=root,
    stdout=logf,
    stderr=subprocess.STDOUT,
    env={**os.environ},
)
print("Uvicorn PID:", proc.pid, "- logs: /tmp/uvicorn_soc.log")
BASE = f"http://127.0.0.1:{PORT}"

# Wait for /healthz (import can take 2–5+ minutes on CPU)
import urllib.request, json
for i in range(120):
    time.sleep(3)
    try:
        with urllib.request.urlopen(BASE + "/healthz", timeout=5) as r:
            body = r.read().decode()
            print("API up:", body[:200])
            break
    except Exception as e:
        if i == 0 or (i + 1) % 5 == 0:
            print(f"... waiting ({i+1}/120):", type(e).__name__)
else:
    print("Server did not become ready. Last part of log:")
    try:
        with open("/tmp/uvicorn_soc.log") as _lf:
            _t = _lf.read()
            print(_t[-4000:] if len(_t) > 4000 else _t)
    except OSError as _e:
        print(_e)

## Call the API from this notebook
These requests go to **127.0.0.1** in the same Colab VM (no tunnel needed).
Open **`/docs`** in the optional tunnel below if you want a browser on another machine.

In [ ]:
import json
import urllib.request

def get(path: str) -> str:
    with urllib.request.urlopen(BASE + path) as r:
        return r.read().decode()

def post_json(path: str, data: dict) -> dict:
    body = json.dumps(data).encode("utf-8")
    req = urllib.request.Request(BASE + path, data=body, headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req) as r:
        return json.loads(r.read().decode())

print(get("/api/datasets/summary"))
print(post_json("/api/train", {
    "episodes": 2,
    "model_name": "distilgpt2",
    "learning_rate": 1e-5,
    "push_to_hub": False,
    "mode": "single_agent",
    "campaign_length": 20,
    "negotiation_rounds": 2,
    "seed": 42,
}))

## Optional: public URL (ngrok)
1. Get a free token at [ngrok.com](https://ngrok.com/) → **Authtoken**.
2. Colab: **Secrets** (key icon) add `NGROK_AUTH_TOKEN`.
3. Run the cell below to open the web UI in a new tab. **Do not** share tokens.

In [ ]:
# Optional public HTTPS URL to the same Uvicorn port (for browser: /, /docs)
# Run the Uvicorn cell first; port must match (default 8000).
import os
import subprocess
import sys

PUBLIC_PORT = 8000
try:
    from google.colab import userdata  # type: ignore
    token = userdata.get("NGROK_AUTH_TOKEN")
except Exception:
    token = os.environ.get("NGROK_AUTH_TOKEN", "")
if not token:
    print("Set Secret NGROK_AUTH_TOKEN or env NGROK_AUTH_TOKEN to use ngrok.")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])
    from pyngrok import ngrok  # type: ignore
    ngrok.set_auth_token(token)
    http_tunnel = ngrok.connect(PUBLIC_PORT, bind_tls=True)
    public = str(http_tunnel.public_url)
    print("Public URL:", public)
    print("API docs:", public + "/docs")